# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and analyze the FAIR² tabular dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is defined by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is available (uncomment the following line if not installed)
!pip install mlcroissant --quiet

## 1. Data Loading
Load dataset metadata and preview the records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset from the URL
dataset = mlc.Dataset(croissant_url)

# The .metadata property exposes the dataset summary as attributes
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets and the fields (columns) available for exploration. All references use the schema entity `@id`s.

In [ ]:
# List available record sets by their @id and display their fields (by @id)
print("Record sets available in the dataset:")
record_sets = dataset.metadata.record_set  # List of RecordSet objects
record_set_ids = []
for rs in record_sets:
    rs_id = getattr(rs, '@id', None)
    record_set_ids.append(rs_id)
    rs_name = getattr(rs, 'name', rs_id)
    print(f"  - {rs_id} (name: {rs_name})")
    if hasattr(rs, 'field') and rs.field:
        field_list = rs.field if isinstance(rs.field, list) else [rs.field]
        print("    Fields (by @id):")
        for f in field_list:
            print(f"      - {getattr(f, '@id', f)} (name: {getattr(f, 'name', f)})")
    print("")

### Example records from each RecordSet
Preview a few records using each record set's `@id`.

In [ ]:
for rs_id in record_set_ids:
    print(f'Preview from RecordSet: {rs_id}')
    try:
        for i, rec in enumerate(dataset.records(record_set=rs_id)):
            print(f"  Record {i+1}: {rec}")
            if i >= 2:
                break
    except Exception as e:
        print(f"  Could not preview records: {e}")
    print("")

## 3. Data Extraction
Extract tabular data from a selected RecordSet (using the RecordSet `@id`).

The typical FAIR² dataset defines a single major record set with all patient/clinical records, but there may be additional sets. Let's choose the main data record set and display columns/fields and a preview.

In [ ]:
# Use all available record sets by their @id for extraction
dataframes = {}

for rs_id in record_set_ids:
    try:
        # Collect all available records into a DataFrame
        records = list(dataset.records(record_set=rs_id))
        if records:
            dataframes[rs_id] = pd.DataFrame(records)
            print(f"RecordSet {rs_id} loaded successfully as a DataFrame.")
        else:
            print(f"RecordSet {rs_id} is empty or could not be loaded.")
    except Exception as e:
        print(f"Failed to extract RecordSet {rs_id}: {e}")

# Pick first non-empty RecordSet for demonstration
main_rs_id = None
for rs_id, df in dataframes.items():
    if not df.empty:
        main_rs_id = rs_id
        break
if main_rs_id:
    print(f"\nMain RecordSet for analysis: {main_rs_id}")
    print("Columns (field @id's):")
    print(dataframes[main_rs_id].columns.tolist())
    dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing steps such as filtering, normalization, and grouping using field `@id`s.

In [ ]:
# We'll select a numeric field for analysis if possible
df = dataframes[main_rs_id]

# Detect numeric-like columns (fields) via dtype
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print(f"Numeric fields detected: {numeric_fields}")
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Using {numeric_field_id} as the numeric field for demonstration.")
    
    # Filter based on a threshold
    threshold = df[numeric_field_id].mean()  # Use mean as a dynamic threshold
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())
    
    # Normalize the numeric field (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Try grouping by a non-numeric field, such as a diagnosis or categorical attribute
    group_field = None
    for col in df.columns:
        if col != numeric_field_id and (df[col].dtype == object or pd.api.types.is_categorical_dtype(df[col])):
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field}:")
        print(grouped_df.head())
else:
    print("No numeric fields found for demonstration.")

## 5. Visualization
Visualize the distribution of a selected numeric field or a relationship between two fields.

If available, use field `@id` for labels.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id and numeric_fields:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
    
    # If we have a group field, create a boxplot
    if group_field:
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=df, x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion
In this notebook, we:
- Loaded metadata and clinical table records from the FAIR² colorectal cancer survivor dataset using `mlcroissant`.
- Explored record sets, their fields (by `@id`), and previewed the data structure.
- Demonstrated basic data extraction, aggregation, normalization, and filtering using field `@id`s.
- Visualized the distribution of a key numeric variable and its relationship with a categorical field if available.

**All references used Croissant `@id`s to allow robust and reproducible data operations that are schema-aware.**

For deeper insights, consider further exploratory data analysis, hypothesis testing, or machine learning using field `@id` references throughout.